# **Simulation Albatross 2026**

Team: TU Wien Space Team \
Project: Lamarr \
Rocket: Albatross


## Installs

In this section all needed libraries are installed and the needed classes imported

In [ ]:
# %pip install rocketpy==1.12.1
# %pip install openmeteo-requests requests-cache retry-requests pandas plotly colorama
# %pip install --upgrade nbformat
# %pip install "niquests==3.18.8" "urllib3-future==2.20.904"

In [ ]:
from rocketpy import (
    Environment,
    Rocket,
    Flight,
    SolidMotor,
    NoseCone,
    Tail,
    Parachute,
    TrapezoidalFins,
    FreeFormFins,
    Function
)
from rocketpy.simulation import FlightDataExporter

import numpy as np
import pandas as pd
from math import pi
import datetime
from pathlib import Path
from colorama import Fore, Style                                  # https://github.com/tartley/colorama


import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parents[1]))   # walk up 1 level to root folder
from simulation.export_weather_data import export_weather_data
from simulation.custom_print_and_plot_functions import CustomPlots, CustomPrints


In [ ]:
# reload imported Python modules when you change their .py files
%load_ext autoreload
%autoreload 2

## Configuration

The length unit chosen here is millimeters to keep the values more readable. If necessary, values should be converted accordingly.

In [ ]:
# Rocket configuration data
tomorrow = datetime.date.today() + datetime.timedelta(days=1)
use_standard_env = False

# TODO: set exact launch rail coordinates and time of launch
env_config = {
    # "date": (2026, 6, 13, 12, 00, 00),                           # yyyy, mm, dd, hh, mm, ss (local time) # preset
    "date": (tomorrow.year, tomorrow.month, tomorrow.day, 8),
    "latitude": 48.802492,                                          # Launch latitude        # preset
    "longitude": 12.550709,                                          # Launch longitude        # preset
    "max_expected_height": 700,                                    # m (AGL)                               # preset    
    "timezone": "Europe/Berlin",                                   # GMT+1                                 # preset
    "altitude": 353.98,                                               # m (ASL)                               # preset
}

motor_config = {
    "name": "K700W",                                                # from https://www.thrustcurve.org/motors/AeroTech/K700W/
    "total_mass": 2035,                                             # g
    "propellant_mass": 1303,                                        # g
    "diameter": 54,                                                 # mm
    "length": 568,                                                  # mm
    "burn_time": 3.5                                                # s
}

rocket_config = {
    "total_mass_without_motor": 17887,                             # g                 # weighed
    "total_length": 3755,                                          # mm                # measured
    "total_CG_without_motor": 1818.79,                             # mm                # measured      # FROM TIP!
    "moment_of_intertia_Z": 0.037801,                              # kg*m^2            # calculated
    "moment_of_intertia_XY": 21.822740,                            # kg*m^2            # calculated
    "nosecone": {
        "length": 659,                                             # mm                # measured    # nosecone (585) + nosetip (74) = 659
        "cylindrical_section_length": 81,                           # mm                # Onshape
        "kind": "lvhaack"
    },
    "railbuttons": {
        "upper": 2385,                                             # mm                # measured      # FROM TIP!
        "lower": 3431                                              # mm                # measured      # FROM TIP!
    },
    "tailcone": {
        "tailcone_bottom_radius": 53.35,                           # mm                # measured
        "cylindrical_section_length": 35,                           # mm                # Onshape
        "length": 280,                                             # mm                # measured
    },
    "rocket": {
        "diameter": 133                                            # mm                # measured
    },
    "fins": {
        "amount": 4,
        "position": 239.36126,                                     # mm                # Onshape (distance of fins coming out of rocket to rear end rocket)
        "shape_points": ((0,0),             # root leading edge    # m                 # measured
                        (0.1928,0.0977),    # up /
                        (0.248,0.0977),     # straight —
                        (0.248,-0.0123)),   # down |               # REMARK no need to repeat (0, 0), rocketpy will connect it automatically
    },
    
    # TODO: need projected area of parachute
    "parachutes": {
        "main": {
            "cd": 1.8,
            "radius": 2.1 / 2,                                      # m                 of projected area
            "trigger": 250,                                        # m                 # preset
            "lag": 5,                                              # s                 # measured
            "sampling_rate": 100,                                  # Hz                # use default value
        },
        "drogue": {    
            "cd": 0.55,                                            
            "radius": 0.4415481,                                   # m     for a circle with area A_fabric (we use fabric since the projected area is almost the one from the fabric)
            "fabric_area": 0.35**2 * 5,                            # m²
            "trigger": "apogee",                                   # m                 # preset        
            "lag": 1,                                              # s                 # measured
            "sampling_rate": 100,                                  # Hz                # use default value
        }
    }
}
flight_config = {
    "rail_length": 7,                                              # m                 # preset
    "inclination": 85,                                             # °                 # preset
    "heading": 184,                                                # °                 # preset
    "terminate_on_apogee": False
}

## Environments Initialization

In this section the environments are initialized.

**Load data from API for custom environment and create CSV for OpenRocket.**\
Docu for custom atmosphere: https://docs.rocketpy.org/en/latest/user/environment/1-atm-models/custom_atmosphere.html

In [ ]:
latitude   = env_config["latitude"]
longitude  = env_config["longitude"]
altitude   = env_config["altitude"]
timezone   = env_config["timezone"]
date       = env_config["date"]
max_expected_height = env_config["max_expected_height"] + altitude      # convert to ASL for the environment
max_expected_height_agl_m = env_config["max_expected_height"]

weather_model_1 = "icon_d2"
weather_model_2 = "icon_eu"

for weather_model in [weather_model_1, weather_model_2]:
    export_weather_data(
        latitude=latitude,
        longitude=longitude,
        max_expected_height_agl_m=max_expected_height,
        launch_time=datetime.datetime(*date).strftime('%Y-%m-%dT%H:%M:%S'),
        timezone=timezone,
        weather_model=weather_model,
    )


**Create environments**
*   **envForecast**: weather data for future date at the specified location
*   **envNormal**: normalized environment with standard atmospheric values at the same time and location
*   **envCustom**: custom weather data from source not provided by RocketPy


Read more about it here https://docs.rocketpy.org/en/latest/user/environment/1-atm-models/index.html


In [ ]:
def print_ev(env):
    # env.prints.gravity_details()
    env.prints.launch_site_details()
    env.prints.atmospheric_model_details()
    env.prints.atmospheric_conditions()
    # env.prints.print_earth_details()
    
    
environments = {}

if use_standard_env:
    # --- Standard Atmosphere Environment ---
    env_Standardized = Environment(max_expected_height=max_expected_height)
    env_Standardized.set_location(latitude=latitude, longitude=longitude)
    # env_Standardized.set_elevation("Open-Elevation")         # API currently not working
    env_Standardized.set_elevation(altitude)
    env_Standardized.set_date(date, timezone=timezone)
    env_Standardized.set_atmospheric_model(type="standard_atmosphere")
    
    print("Standard Atmosphere Environment:")
    print_ev(env_Standardized)
    env_Standardized.plots.atmospheric_model()
    environments["Standardized"] = env_Standardized

else:
    # --- Forecast Environment ---
    env_ICONEU_WINDY = Environment(max_expected_height=max_expected_height)
    env_ICONEU_WINDY.set_location(latitude=latitude, longitude=longitude)
    # env_ICONEU_WINDY.set_elevation("Open-Elevation")         # API currently not working
    env_ICONEU_WINDY.set_elevation(altitude)
    env_ICONEU_WINDY.set_date(date, timezone=timezone)
    env_ICONEU_WINDY.set_atmospheric_model(type="Windy", file="ICONEU")
    
    print("Forecast Environment:")
    print_ev(env_ICONEU_WINDY)
    env_ICONEU_WINDY.plots.atmospheric_model()
    environments["ICONEU_WINDY"] = env_ICONEU_WINDY


    # --- Custom Data Environment ---
    # Load the .csv file into the environment https://docs.rocketpy.org/en/latest/user/environment/3-further/data_csv.html#load-the-csv-file
    output_folder = Path("./weather_csvs")
    output_folder.mkdir(exist_ok=True)
    for weather_model in [weather_model_1, weather_model_2]:
        name = f"Custom_{weather_model}"
        df = pd.read_csv(output_folder / f'rocketpy_atmosphere_{weather_model}.csv')

        # Create Function objects to represent the profiles
        pressure_func = Function(np.column_stack([df['height'], df['pressure']]))
        temperature_func = Function(np.column_stack([df['height'], df['temperature']]))
        wind_u_func = Function(np.column_stack([df['height'], df['wind_u']]))
        wind_v_func = Function(np.column_stack([df['height'], df['wind_v']]))

        # Set up the environment
        env = Environment(max_expected_height=max_expected_height)
        env.set_location(latitude=latitude, longitude=longitude)
        # env.set_elevation("Open-Elevation")         # API currently not working
        env.set_elevation(altitude)
        env.set_date(date, timezone=timezone)
        env.set_atmospheric_model(
            type="custom_atmosphere",
            pressure=pressure_func,
            temperature=temperature_func,
            wind_u=wind_u_func,
            wind_v=wind_v_func,
        )

        print(f"Custom Environment [{weather_model}]:")
        print_ev(env)
        env.plots.atmospheric_model()
        environments[name] = env

## Simulation
### Engine
https://docs.rocketpy.org/en/latest/user/motors/solidmotor.html

In [ ]:
motor_total_mass        = motor_config["total_mass"] / 1000                             # kg
motor_propellant_mass   = motor_config["propellant_mass"] / 1000                        # kg
motor_dry_mass          = motor_total_mass - motor_propellant_mass                      # kg
motor_radius            = motor_config["diameter"] / 1000 / 2                           # m
motor_length            = motor_config["length"]  / 1000                                # m
motor_volume            = pi * (motor_radius ** 2) * motor_length                       # m³
motor_grain_density     = motor_propellant_mass / motor_volume                          # kg/m³
motor_burn_time         = motor_config["burn_time"]
solid_motor = True

# inertia of motor without propellant (dry mass) using formula for thin cylindrical shell with open ends
Ix = Iy = 1/12 * motor_dry_mass * (6* motor_radius **2 + motor_length**2)               # kg*m²
Iz = motor_dry_mass * motor_radius **2                                                  # kg*m²

# TODO: there might be a dataset of motors somewhere from rocketpy
k700w = SolidMotor(
    thrust_source                   = "AeroTech_K700W.eng",            # can be .eng/.rsa file for commercial motor or .csv file from static fire test
    dry_mass                        = motor_dry_mass,
    dry_inertia                     = (Ix, Iy, Iz),                     # not so important, but we still could do it percise
    nozzle_radius                   = motor_radius,
    grain_number                    = 1,
    grain_density                   = motor_grain_density,
    grain_outer_radius              = motor_radius,
    grain_initial_inner_radius      = 0,
    grain_initial_height            = motor_length,
    grain_separation                = 0,
    grains_center_of_mass_position  = motor_length / 2,
    center_of_dry_mass_position     = motor_length * 0.4,           # due to nozzle at the rear end
    nozzle_position                 = 0,
    burn_time                       = motor_burn_time,
    throat_radius                   = motor_radius / 2,
    coordinate_system_orientation   = "nozzle_to_combustion_chamber",
)

# --- prints ---
print(Fore.CYAN + "\n== Inertia of motor without propellant ==" + Style.RESET_ALL)
print(f"Ix={Ix}, Iy={Iy}, Iz={Iz}\n")
k700w.prints.nozzle_details()
k700w.prints.motor_details()

if solid_motor:
    k700w.prints.grain_details()
    
# --- plots ---
k700w.draw()
k700w.plots.thrust()
# k700w.plots.mass_flow_rate()
# k700w.plots.exhaust_velocity()
k700w.plots.total_mass()
# k700w.plots.propellant_mass()
k700w.plots.center_of_mass()  
# k700w.plots.burn_rate()
# k700w.plots.burn_area()
# k700w.plots.Kn()
k700w.plots.inertia_tensor()

# if solid_motor:
    # k700w.plots.grain_inner_radius()
    # k700w.plots.grain_height()


### Rocket components

In [ ]:
# rocket
rocket_length     = rocket_config["total_length"]         / 1000
rocket_diameter   = rocket_config["rocket"]["diameter"]   / 1000

nosecone_length = (rocket_config["nosecone"]["length"] - rocket_config["nosecone"]["cylindrical_section_length"]) / 1000
nosecone_kind = rocket_config["nosecone"]["kind"]

nose_cone = NoseCone(
    length=nosecone_length, 
    base_radius=rocket_diameter / 2, 
    kind=nosecone_kind
)

tailcone_length         = (rocket_config["tailcone"]["length"] - rocket_config["tailcone"]["cylindrical_section_length"]) / 1000
tailcone_bottom_radius  = rocket_config["tailcone"]["tailcone_bottom_radius"]       / 1000

tail = Tail(
    top_radius=rocket_diameter / 2,
    bottom_radius=tailcone_bottom_radius,
    length=tailcone_length,
    rocket_radius=rocket_diameter / 2,
)

fin_amount        = rocket_config["fins"]["amount"]
fin_shape_points  = rocket_config["fins"]["shape_points"]

fin_set = FreeFormFins(
    n=fin_amount,
    shape_points=fin_shape_points,
    rocket_radius=tailcone_bottom_radius,               # TODO: create airfoil file and use it here https://docs.rocketpy.org/en/latest/user/rocket/rocket_usage.html#adding-airfoil-profile-to-fins
)
fin_set.draw()


# compute area with https://en.wikipedia.org/wiki/Shoelace_formula
Af = 0
# go through points P1...Pn
for i in range(len(fin_set.shape_points) - 1):
    x1, y1 = fin_set.shape_points[i]
    x2, y2 = fin_set.shape_points[i + 1]
    Af += (y1 + y2) * (x1 - x2)
# close the polygon
x1 = fin_set.shape_points[0][0]
xn = fin_set.shape_points[-1][0] 
y1 = fin_set.shape_points[0][1]
yn = fin_set.shape_points[-1][1]      
Af += (yn + y1) * (xn - x1)

# sign depends on order the points are provided
Af = abs(Af) / 2
print(f"Fin area: {Af}")
print(f"Fin area RocketPy: {fin_set.Af}")
        
# fin_set.prints.identity()
fin_set.prints.geometry()
# fin_set.prints.lift()
# fin_set.plots.airfoil()
# fin_set.plots.roll()
# fin_set.plots.lift()


# fin_set = TrapezoidalFins(
#       n             = fin_amount,
#       root_chord    = 0.250,
#       tip_chord     = 0.050,
#       span          = 0.108,
#       sweep_length  = 0.200,
#       rocket_radius = tailcone_bottom_radius,
#       name          = "Trapezoidal"
# )
# fin_set.draw()

main_cd                 = rocket_config["parachutes"]["main"]["cd"]
main_radius             = rocket_config["parachutes"]["main"]["radius"]
main_trigger            = rocket_config["parachutes"]["main"]["trigger"]
main_sampling_rate      = rocket_config["parachutes"]["main"]["sampling_rate"]
main_lag                = rocket_config["parachutes"]["main"]["lag"]
# main_noise              = rocket_config["parachutes"]["main"]["noise"]
main_cd_s               = main_cd * pi * main_radius**2                              # C_D * A_projected 

drogue_cd               = rocket_config["parachutes"]["drogue"]["cd"]
drogue_radius           = rocket_config["parachutes"]["drogue"]["radius"]
drogue_fabric_area      = rocket_config["parachutes"]["drogue"]["fabric_area"]
drogue_trigger          = rocket_config["parachutes"]["drogue"]["trigger"]
drogue_sampling_rate    = rocket_config["parachutes"]["drogue"]["sampling_rate"]
drogue_lag              = rocket_config["parachutes"]["drogue"]["lag"]
# drogue_noise            = rocket_config["parachutes"]["drogue"]["noise"]
drogue_cd_s             = drogue_cd * drogue_fabric_area

parachutes = {}

parachutes[0] = Parachute(
    name="main",
    cd_s=main_cd_s,
    trigger=main_trigger,
    lag=main_lag,
    radius=main_radius,
    drag_coefficient=main_cd,
    sampling_rate=main_sampling_rate,
)

parachutes[1] = Parachute(
    name="drogue",
    cd_s=drogue_cd_s,
    trigger=drogue_trigger,
    lag=drogue_lag,
    radius=drogue_radius,
    drag_coefficient=drogue_cd,
    sampling_rate=drogue_sampling_rate,
)


for parachute in parachutes.values():
    CustomPlots.plot_parachute_model(parachute)


### ALBATROSS
RocketPy definitions:
- dry mass = rocket with motor but without propellant
- Rocket Loaded Mass = Wet mass
- Rocket Center of Dry Mass - Nozzle Exit = Rocket Center of Dry Mass from bottom


In [ ]:
total_mass_without_motor = rocket_config["total_mass_without_motor"] / 1000

# inertia
inertia_x_y   = rocket_config["moment_of_intertia_XY"]
inertia_z     = rocket_config["moment_of_intertia_Z"]

upper_railbutton_position_from_bottom   = rocket_length - rocket_config["railbuttons"]["upper"] / 1000      # convert to from bottom
lower_railbutton_position_from_bottom   = rocket_length - rocket_config["railbuttons"]["lower"] / 1000
fin_position                            = rocket_config["fins"]["position"]     / 1000

total_CG_without_motor_from_bottom = rocket_length - rocket_config["total_CG_without_motor"] / 1000     # convert to from bottom

albatross = Rocket(
    radius=rocket_diameter / 2,
    mass=total_mass_without_motor,
    inertia=(inertia_x_y, inertia_x_y, inertia_z),
    power_off_drag="./power_off_drag.csv",
    power_on_drag="./power_on_drag.csv",
    center_of_mass_without_motor=total_CG_without_motor_from_bottom,                        
    coordinate_system_orientation="tail_to_nose",
)


albatross.add_motor(k700w, position=0)
albatross.set_rail_buttons(upper_button_position=upper_railbutton_position_from_bottom, lower_button_position=lower_railbutton_position_from_bottom)
albatross.add_surfaces(surfaces=[nose_cone, fin_set, tail], positions=[rocket_length, fin_position, tailcone_length])
albatross.parachutes = list(parachutes.values())


# --- prints ---
print(f"Rocket center of wet mass from tip: {(rocket_length - albatross.center_of_mass(0)) * 1000} mm")

albatross.prints.inertia_details()
# albatross.prints.rocket_geometrical_parameters()
# albatross.prints.rocket_aerodynamics_quantities()
albatross.prints.parachute_data()

# --- plots ---
albatross.plots.draw()
albatross.plots.total_mass()
# albatross.plots.reduced_mass()
albatross.plots.drag_curves()
# albatross.plots.static_margin()
# albatross.plots.stability_margin()
albatross.plots.thrust_to_weight()

### Flight

In [ ]:
rail_length           = flight_config["rail_length"]
inclination           = flight_config["inclination"]
heading               = flight_config["heading"]
terminate_on_apogee   = flight_config["terminate_on_apogee"]
flight_forecasts = {}

output_folder = Path("./trajectory_kml")
output_folder.mkdir(exist_ok=True)
    
for env_name, env in environments.items():
    print(Fore.GREEN + f"\n\n--- Simulating Flight in Environment: {env_name} ---" + Style.RESET_ALL)
    flight_forecast = Flight(
        rocket=albatross,
        environment=env,
        rail_length=rail_length,
        inclination=inclination,
        heading=heading,
        terminate_on_apogee=terminate_on_apogee,
        name=env_name,
    )
    
    # --- prints ---
    # flight_forecast.prints.all()
    custom_prints = CustomPrints(flight_forecast)    
    flight_forecast.prints.launch_rail_conditions()
    flight_forecast.prints.out_of_rail_conditions()
    custom_prints.apogee_conditions()
    # flight_forecast.prints.apogee_conditions()
    custom_prints.parachute_events()
    # flight_forecast.prints.events_registered()
    flight_forecast.prints.impact_conditions()
    custom_prints.impact_coordinates()
    # flight_forecast.prints.maximum_values()

    
    # --- plots --- 
    # flight_forecast.plots.all()
    custom_plots = CustomPlots(
        flight_forecast=flight_forecast,
        motor=k700w,
        plot_title=env_name,
        rocket=albatross,
        rocket_config=rocket_config,
    )
    
    custom_plots.plot_stability_and_cg_cp_position()
    # flight_forecast.plots.stability_and_control_data()
    custom_plots.plot_angle_of_attack()
    custom_plots.plot_vertical_motion()
    flight_forecast.plots.trajectory_3d()
    
    
    file_name = output_folder / f"ALBATROSS_Flight_Forecast_{env_name}.kml"
    FlightDataExporter(flight_forecast).export_kml(file_name=file_name, altitude_mode="relativetoground")
    flight_forecasts[env_name] = flight_forecast
